In [ ]:
"""
LLM Client with Caching

This module provides an LLM client that caches responses to prevent
repeated API calls during testing.
"""

import os
import json
import hashlib
from typing import Optional
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()


def _get_cache_file(input_hash: str) -> str:
    """Get cache file path based on input content hash for persistence across runs."""
    cache_dir = os.path.join(os.getcwd(), ".pytest_cache")
    os.makedirs(cache_dir, exist_ok=True)
    return os.path.join(cache_dir, f"cache_{input_hash[:8]}.json")


def _get_input_hash(prompt: str, model: str, max_tokens: int, tools: Optional[list] = None) -> str:
    """Create a hash of input parameters for caching."""
    tools_str = json.dumps(tools, sort_keys=True) if tools else ""
    input_str = f"{prompt}|{model}|{max_tokens}|{tools_str}"
    return hashlib.sha256(input_str.encode()).hexdigest()


class LLMClient:
    """Simple LLM client for chat completions with function calling and caching"""

    def __init__(self):
        """Initialize LLM client with environment validation"""
        self._validate_environment()
        self._client = None

    def _validate_environment(self):
        """Validate required environment variables"""
        api_key = os.getenv("OPENAI_API_KEY")
        if not api_key:
            raise ValueError(
                "OpenAI API key is required for LLM operations. "
                "Please set environment variables:\n"
                "OPENAI_API_KEY=your_api_key\n"
                "OPENAI_API_BASE=https://api.openai.com/v1 (optional)\n"
                "Example: export OPENAI_API_KEY=sk-your-key"
            )

    @property
    def client(self) -> OpenAI:
        """Get OpenAI client instance (lazy initialization)"""
        if self._client is None:
            api_key = os.getenv("OPENAI_API_KEY")
            base_url = os.getenv("OPENAI_API_BASE")

            self._client = OpenAI(
                api_key=api_key,
                base_url=base_url if base_url else None
            )
        return self._client

    def chat_completion(self, messages: list, tools: Optional[list] = None, model: str = "gpt-4o-mini", max_tokens: int = 500):
        """
        Create a chat completion with optional function calling

        Args:
            messages: List of message dictionaries
            tools: Optional list of tool schemas for function calling
            model: Model to use (default: gpt-4o-mini)
            max_tokens: Maximum tokens in response (default: 500)

        Returns:
            The OpenAI response object
        """
        cache_key = _get_input_hash(str(messages), model, max_tokens, tools)
        cache_file = _get_cache_file(cache_key)

        try:
            if os.path.exists(cache_file):
                with open(cache_file, "r", encoding="utf-8") as f:
                    cache_data = json.load(f)
                    if cache_key in cache_data:
                        # Reconstruct response object from cache
                        cached = cache_data[cache_key]
                        return self._reconstruct_response(cached)
        except (json.JSONDecodeError, IOError, OSError):
            pass

        try:
            params = {
                "model": model,
                "messages": messages,
                "max_tokens": max_tokens,
                "temperature": 0.1
            }
            if tools:
                params["tools"] = tools

            response = self.client.chat.completions.create(**params)

            # Cache the response
            try:
                cache_data = {}
                if os.path.exists(cache_file):
                    with open(cache_file, "r", encoding="utf-8") as f:
                        cache_data = json.load(f)

                # Store response data
                response_data = {
                    "choices": [{
                        "message": {
                            "role": response.choices[0].message.role,
                            "content": response.choices[0].message.content,
                            "tool_calls": [
                                {
                                    "id": tc.id,
                                    "type": tc.type,
                                    "function": {
                                        "name": tc.function.name,
                                        "arguments": tc.function.arguments
                                    }
                                } for tc in (response.choices[0].message.tool_calls or [])
                            ]
                        },
                        "finish_reason": response.choices[0].finish_reason
                    }]
                }

                cache_data[cache_key] = response_data

                with open(cache_file, "w", encoding="utf-8") as f:
                    json.dump(cache_data, f, indent=2)
            except Exception:
                pass

            return response
        except Exception as e:
            raise Exception(f"LLM completion failed: {e}") from e

    def _reconstruct_response(self, cached_data: dict):
        """Reconstruct a response-like object from cached data"""
        class CachedResponse:
            def __init__(self, data):
                self.choices = [CachedChoice(data["choices"][0])]

        class CachedChoice:
            def __init__(self, data):
                self.message = CachedMessage(data["message"])
                self.finish_reason = data.get("finish_reason")

        class CachedMessage:
            def __init__(self, data):
                self.role = data.get("role")
                self.content = data.get("content")
                tool_calls_data = data.get("tool_calls", [])
                if tool_calls_data:
                    self.tool_calls = [CachedToolCall(tc) for tc in tool_calls_data]
                else:
                    self.tool_calls = None

        class CachedToolCall:
            def __init__(self, data):
                self.id = data["id"]
                self.type = data["type"]
                self.function = CachedFunction(data["function"])

        class CachedFunction:
            def __init__(self, data):
                self.name = data["name"]
                self.arguments = data["arguments"]

        return CachedResponse(cached_data)


_llm_client: Optional[LLMClient] = None


def get_llm_client() -> LLMClient:
    """Get the global LLM client instance (singleton pattern)"""
    global _llm_client
    if _llm_client is None:
        _llm_client = LLMClient()
    return _llm_client

In [ ]:
"""
Tool Schema Definitions

This module defines tool schemas for OpenAI function calling.
You need to fix vague descriptions and incorrect parameter types.
"""

from typing import List, Dict


def get_weather_schema() -> Dict:
    """
    Returns the schema for the get_weather tool.
    """
    return {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Gets the current weather for a specific location.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The location to get the weather for."
                    },
                    "units": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "The unit of temperature to use."
                    }
                },
                "required": ["location"]
            }
        }
    }


def format_weather_response_schema() -> Dict:
    """
    Returns the schema for the format_weather_response tool.
    """
    return {
        "type": "function",
        "function": {
            "name": "format_weather_response",
            "description": "Formats weather response data for presentation.",
            "parameters": {
                "type": "object",
                "properties": {
                    "weather_data": {
                        "type": "object",
                        "description": "The weather data to format."
                    },
                    "style": {
                        "type": "string",
                        "enum": ["brief", "detailed", "technical"],
                        "description": "The style of the formatted response."
                    }
                },
                "required": ["weather_data"]
            }
        }
    }


def get_all_schemas() -> List[Dict]:
    """
    Returns a list of all tool schemas.

    Returns:
        List of tool schema dictionaries
    """
    return [
        get_weather_schema(),
        format_weather_response_schema()
    ]


In [ ]:
"""
Tool Executor

This module implements the execution layer for tool calls.
You need to implement the execute method and error handling.
"""

from typing import Dict, Any, Optional


class ToolExecutor:
    """
    Executes tool calls by routing to appropriate implementations.
    """

    def __init__(self):
        """Initialize the tool executor with available tools"""
        self.tools = {
            "get_weather": self._get_weather,
            "format_weather_response": self._format_weather_response
        }

    def execute(self, function_name: str, arguments: Dict[str, Any]) -> Any:
        """
        Execute a tool call by routing to the appropriate implementation.

        Args:
            function_name: Name of the function to execute
            arguments: Dictionary of arguments for the function

        Returns:
            The result from the tool function, or a dict with an error key.
        """
        if function_name not in self.tools:
            return {"error": f"Unknown function: {function_name}"}

        try:
            return self.tools[function_name](**arguments)
        except Exception as e:
            return {"error": str(e)}

    def _get_weather(self, location: str, units: str = "celsius") -> Dict[str, Any]:
        """
        Mock implementation of get_weather tool.

        Args:
            location: Location to get weather for
            units: Temperature units ("celsius" or "fahrenheit")

        Returns:
            Dictionary with weather data
        """
        base_temp = 22 if units == "celsius" else 72

        return {
            "location": location,
            "temperature": base_temp,
            "condition": "sunny",
            "humidity": 65,
            "units": units
        }

    def _format_weather_response(self, weather_data: Dict[str, Any], style: str = "brief") -> str:
        """
        Mock implementation of format_weather_response tool.

        Args:
            weather_data: Dictionary with weather information
            style: Format style ("brief", "detailed", or "technical")

        Returns:
            Formatted weather string
        """
        location = weather_data.get("location", "Unknown")
        temp = weather_data.get("temperature", 0)
        units = weather_data.get("units", "celsius")
        condition = weather_data.get("condition", "unknown")

        if style == "brief":
            return f"Weather in {location}: {temp}°{units[0].upper()}, {condition}"
        elif style == "detailed":
            humidity = weather_data.get("humidity", 0)
            return f"Current weather in {location}: Temperature {temp}°{units[0].upper()}, Condition: {condition}, Humidity: {humidity}%"
        else:
            return f"{{location: '{location}', temp: {temp}, units: '{units}', condition: '{condition}'}}"

Weather in Tokyo: 25°C, cloudy


In [ ]:
"""
Weather Agent with Function Calling

This module implements an agent that uses OpenAI function calling
to interact with weather tools.
"""

from typing import Dict, List, Optional
import json
from tool_schemas import get_all_schemas
from tool_executor import ToolExecutor
from llm import get_llm_client


class WeatherAgent:
    """
    Agent that uses function calling to get and format weather information.
    """

    def __init__(self, tool_executor: Optional[ToolExecutor] = None):
        """
        Initialize the weather agent.

        Args:
            tool_executor: Optional tool executor instance (creates new one if not provided)
        """
        self.executor = tool_executor or ToolExecutor()
        self.llm = get_llm_client()
        self.schemas = get_all_schemas()

    def process_request(self, user_request: str) -> str:
        """
        Process a user request using function calling.

        Args:
            user_request: User's request (e.g., "What's the weather in Paris?")

        Returns:
            Final response string from the agent
        """
        messages = [{"role": "user", "content": user_request}]
        tools = self.schemas

        for _ in range(3):  # Max 3 iterations
            result = self.llm.chat_completion(
                messages=messages,
                tools=tools
            )

            message = result.choices[0].message

            tool_calls = message.tool_calls
            if not tool_calls:
                return message.content or ""

            messages.append({
                "role": "assistant",
                "content": message.content,
                "tool_calls": [
                    {
                        "id": tc.id,
                        "type": tc.type,
                        "function": {
                            "name": tc.function.name,
                            "arguments": tc.function.arguments
                        }
                    }
                    for tc in tool_calls
                ]
            })

            for tc in tool_calls:
                function_name = tc.function.name
                arguments = json.loads(tc.function.arguments)

                tool_result = self.executor.execute(function_name, arguments)

                if isinstance(tool_result, dict):
                    tool_result_str = json.dumps(tool_result)
                else:
                    tool_result_str = str(tool_result)

                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": tool_result_str
                })

        final = self.llm.chat_completion(messages=messages)
        return final.choices[0].message.content or ""